In [ ]:
import pyspark
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("eda").getOrCreate()

from pyspark.sql.types import *

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [13]:
schema = StructType([
    StructField('time',IntegerType(),True),
    StructField('src_user',StringType(),True),
    StructField('dest_user',StringType(),True),
    StructField('src_comp',StringType(),True),
    StructField('dest_comp',StringType(),True),
    StructField('auth_type',StringType(),True),
    StructField('logon_type',StringType(),True),
    StructField('auth_orientation',StringType(),True),
    StructField('outcome',StringType(),True),
])

In [15]:
df = spark.read.csv('/content/drive/MyDrive/lanl local file.csv', header=True, schema=schema)
df.show(5)

+----+--------------------+--------------------+--------+---------+---------+----------+----------------+-------+
|time|            src_user|           dest_user|src_comp|dest_comp|auth_type|logon_type|auth_orientation|outcome|
+----+--------------------+--------------------+--------+---------+---------+----------+----------------+-------+
|   1|ANONYMOUS LOGON@C586|ANONYMOUS LOGON@C586|   C1250|     C586|     NTLM|   Network|           LogOn|Success|
|   1|ANONYMOUS LOGON@C586|ANONYMOUS LOGON@C586|    C586|     C586|        ?|   Network|          LogOff|Success|
|   1|          C101$@DOM1|          C101$@DOM1|    C988|     C988|        ?|   Network|          LogOff|Success|
|   1|         C1020$@DOM1|        SYSTEM@C1020|   C1020|    C1020|Negotiate|   Service|           LogOn|Success|
|   1|         C1021$@DOM1|         C1021$@DOM1|   C1021|     C625| Kerberos|   Network|           LogOn|Success|
+----+--------------------+--------------------+--------+---------+---------+----------+

In [31]:
from pyspark.sql.functions import countDistinct

## Count rows
print(df.count())

# Distinct number of source computers 
print(df.select(countDistinct("src_comp")).collect()[0][0])

# Distinct number of destination computers
print(df.select(countDistinct("dest_comp")).collect()[0][0])

100000
3971
2401


In [ ]:
from pyspark.sql.functions import col, contains

# Machine count
print(df.filter(col('src_user').contains('$')).count())

## Proportion of machines
print(df.filter(col('src_user').contains('$')).count()/df.count())

72457
0.72457


In [ ]:
# Number of success & failures 
df.groupBy('outcome').count().show()

+-------+-----+
|outcome|count|
+-------+-----+
|Success|99257|
|   Fail|  743|
+-------+-----+



In [ ]:
# Number of success and failures for machines
df.filter(col('src_user').contains('$')).groupBy('outcome').count().show()

+-------+-----+
|outcome|count|
+-------+-----+
|Success|72099|
|   Fail|  358|
+-------+-----+



In [ ]:
# Distribution of human source users
df.filter(col('src_user').startswith('U')).groupBy('src_user').count().orderBy('count',ascending = False).show()

+---------+-----+
| src_user|count|
+---------+-----+
| U22@DOM1| 2674|
| U66@DOM1| 1941|
|U292@DOM1|  827|
|  U6@DOM1|  543|
| U63@DOM1|  540|
|  U7@DOM1|  517|
| U94@DOM1|  509|
| U19@DOM1|  504|
| U78@DOM1|  494|
| U14@DOM1|  478|
| U75@DOM1|  477|
| U34@DOM1|  433|
| U48@DOM1|  406|
|  U3@DOM1|  297|
|  U4@DOM1|  260|
|U109@DOM1|  219|
|U113@DOM1|  219|
| U13@DOM1|  217|
| U53@DOM1|  208|
| U24@DOM1|  199|
+---------+-----+
only showing top 20 rows


In [ ]:
# Distribution of destination users
df.groupBy("dest_user").count().orderBy("count", ascending=False).show(20)

+--------------------+-----+
|           dest_user|count|
+--------------------+-----+
|            U22@DOM1| 2674|
|            U66@DOM1| 1941|
|          C599$@DOM1| 1828|
|ANONYMOUS LOGON@C586| 1750|
|         C1114$@DOM1| 1413|
|          C585$@DOM1| 1298|
|          C104$@DOM1| 1296|
|          C743$@DOM1| 1282|
|          C567$@DOM1| 1024|
|          C123$@DOM1|  977|
|         C1617$@DOM1|  848|
|           U292@DOM1|  834|
|          C538$@DOM1|  806|
|         C1065$@DOM1|  778|
|         C1794$@DOM1|  703|
|          C529$@DOM1|  684|
|         C1766$@DOM1|  679|
|          C480$@DOM1|  638|
|          C612$@DOM1|  618|
|             U6@DOM1|  543|
+--------------------+-----+
only showing top 20 rows
